# PDSA Name Matching and Cleansing Framework

## MSc Data Science Project

### Organisation
People's Dispensary for Sick Animals (PDSA)

---

# Project Background

Poor quality name data can significantly affect data integrity, customer identification, reporting and system interoperability. Names often contain typographical errors, abbreviations, nicknames, multilingual spellings, accented characters and inconsistent formatting that reduce the effectiveness of traditional exact matching techniques.

This project develops a Python-based name matching and cleansing framework capable of identifying valid names, correcting inconsistencies, discovering variants and generating canonical names. The final output is designed for future implementation within a SQL database environment.

---

# Project Aim

To design and develop an intelligent name matching framework capable of improving name quality using deterministic rules and approximate string matching techniques.

---

# Objectives

The framework will:

- Profile raw name datasets
- Clean inconsistent data
- Normalise names
- Build canonical dictionaries
- Detect spelling variants
- Apply deterministic business rules
- Apply approximate string matching
- Generate confidence scores
- Produce SQL-ready match tables

---

# Overall Workflow

Raw Data

↓

Data Profiling

↓

Data Cleaning

↓

Name Normalisation

↓

Rule-Based Standardisation

↓

Variant Discovery

↓

Candidate Generation

↓

Matching Engine

↓

Confidence Scoring

↓

Master Match Matrix

↓

SQL Prototype

In [1]:
# ============================================================
# SECTION 2 - ImportING Required Libraries
# ============================================================

# Data manipulation
import pandas as pd
import numpy as np

# Regular expressions
import re

# Unicode and accent handling
from unidecode import unidecode

# String similarity
from rapidfuzz import process, fuzz

# Distance metrics
import jellyfish

# Visualisation
import matplotlib.pyplot as plt

# Utilities
from collections import Counter

# Notebook settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("All libraries loaded successfully.")

All libraries loaded successfully.


In [2]:
# ============================================================
# SECTION 3 - Project Configuration
# ============================================================

# Display options
pd.set_option("display.width", 200)

# Random seed for reproducibility
np.random.seed(42)

# Similarity thresholds
EXACT_MATCH = 100
HIGH_CONFIDENCE = 90
MEDIUM_CONFIDENCE = 80
LOW_CONFIDENCE = 70

print("Project configuration loaded.")

Project configuration loaded.


# SECTION 4 – Data Loading

## Purpose

This section imports the raw datasets used throughout the project.

Two datasets are used:

- **Forenames Dataset** – Contains validated first names and associated metadata.
- **Surnames Dataset** – Contains validated family names and associated metadata.

At this stage, no cleaning or transformation is performed. The objective is to verify that the files have been loaded successfully and to understand their structure before profiling and preprocessing.

In [3]:
# ============================================================
# SECTION 4 - Loading Datasets
# ============================================================

# Importing pandas library
import pandas as pd

# Loading datasets with proper encoding to handle non-UTF-8 characters
forenames = pd.read_csv("forenames.csv", encoding='latin-1')  # Added encoding parameter
surnames = pd.read_csv("surnames.csv", encoding='latin-1')    # Added encoding parameter

print("Datasets loaded successfully.\n")

print(f"Forenames : {forenames.shape[0]:,} rows × {forenames.shape[1]} columns")
print(f"Surnames  : {surnames.shape[0]:,} rows × {surnames.shape[1]} columns")

C:\Users\omoto\AppData\Local\Temp\ipykernel_41096\4289887268.py:9: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  forenames = pd.read_csv("forenames.csv", encoding='latin-1')  # Added encoding parameter


Datasets loaded successfully.

Forenames : 204,108 rows × 9 columns
Surnames  : 4,153,336 rows × 7 columns


In [4]:
print(forenames.columns)
print(surnames.columns)

Index(['FORENAME_UPPER', 'FORENAME_PROPER', 'FORENAME_MATCH', 'EDIT_DISTANCE', 'JARO_WINKLER', 'FORENAME_FLAG', 'AMOUNT', 'GENDER', 'GENDER_COUNT'], dtype='object')
Index(['SURNAME_UPPER', 'SURNAME_PROPER', 'SURNAME_MATCH', 'EDIT_DISTANCE', 'JARO_WINKLER', 'SURNAME_FLAG', 'AMOUNT'], dtype='object')


In [5]:
forenames.head()

,FORENAME_UPPER,FORENAME_PROPER,FORENAME_MATCH,EDIT_DISTANCE,JARO_WINKLER,FORENAME_FLAG,AMOUNT,GENDER,GENDER_COUNT
0,JOHN,John,NaN,0,1.0,G,47638097,M,1
1,DAVID,David,NaN,0,1.0,G,44716008,M,1
2,JAMES,James,NaN,0,1.0,G,29351116,M,1
3,MICHAEL,Michael,NaN,0,1.0,G,29168166,M,1
4,PAUL,Paul,NaN,0,1.0,G,28155602,M,1


In [6]:
surnames.head()

,SURNAME_UPPER,SURNAME_PROPER,SURNAME_MATCH,EDIT_DISTANCE,JARO_WINKLER,SURNAME_FLAG,AMOUNT
0,SMITH,Smith,NaN,0,1.0,G,39891647
1,JONES,Jones,NaN,0,1.0,G,29317272
2,WILLIAMS,Williams,NaN,0,1.0,G,20497885
3,BROWN,Brown,NaN,0,1.0,G,18821086
4,TAYLOR,Taylor,NaN,0,1.0,G,18390861


In [7]:
########################################################
# CREATE WORKING DATAFRAMES
########################################################

df_forenames = forenames.copy()

df_surnames = surnames.copy()

In [8]:
df_forenames["Original_Name"] = df_forenames["FORENAME_PROPER"]

df_forenames["NORMALISED_NAME"] = (
    df_forenames["FORENAME_UPPER"]
    .astype(str)
    .str.strip()
)

In [9]:
df_forenames.head()

,FORENAME_UPPER,FORENAME_PROPER,FORENAME_MATCH,EDIT_DISTANCE,JARO_WINKLER,FORENAME_FLAG,AMOUNT,GENDER,GENDER_COUNT,Original_Name,NORMALISED_NAME
0,JOHN,John,NaN,0,1.0,G,47638097,M,1,John,JOHN
1,DAVID,David,NaN,0,1.0,G,44716008,M,1,David,DAVID
2,JAMES,James,NaN,0,1.0,G,29351116,M,1,James,JAMES
3,MICHAEL,Michael,NaN,0,1.0,G,29168166,M,1,Michael,MICHAEL
4,PAUL,Paul,NaN,0,1.0,G,28155602,M,1,Paul,PAUL


# SECTION 5 – Data Profiling

## Purpose

Before performing any cleaning or matching, it is important to understand the quality and characteristics of the datasets.

This section explores:

- Dataset dimensions
- Data types
- Missing values
- Duplicate records
- Memory usage
- Descriptive statistics
- Name length distribution
- Character composition

The results provide a baseline for measuring the effectiveness of subsequent data cleansing and matching operations.

In [10]:
# ============================================================
# SECTION 5.1 - Dataset Summary
# ============================================================

summary = pd.DataFrame({

    "Dataset":[
        "Forenames",
        "Surnames"
    ],

    "Rows":[
        len(forenames),
        len(surnames)
    ],

    "Columns":[
        forenames.shape[1],
        surnames.shape[1]
    ],

    "Memory (MB)":[
        round(forenames.memory_usage(deep=True).sum()/1024**2,2),
        round(surnames.memory_usage(deep=True).sum()/1024**2,2)
    ]

})

summary

,Dataset,Rows,Columns,Memory (MB)
0,Forenames,204108,9,55.47
1,Surnames,4153336,7,911.06


In [11]:
# ============================================================
# SECTION 5.2 - Data Types
# ============================================================

print("FORNAMES DATA TYPES")
display(forenames.dtypes)

print("\n")

print("SURNAMES DATA TYPES")
display(surnames.dtypes)

FORNAMES DATA TYPES


FORENAME_UPPER      object
FORENAME_PROPER     object
FORENAME_MATCH      object
EDIT_DISTANCE        int64
JARO_WINKLER       float64
FORENAME_FLAG       object
AMOUNT               int64
GENDER              object
GENDER_COUNT         int64
dtype: object



SURNAMES DATA TYPES


SURNAME_UPPER      object
SURNAME_PROPER     object
SURNAME_MATCH      object
EDIT_DISTANCE       int64
JARO_WINKLER      float64
SURNAME_FLAG       object
AMOUNT              int64
dtype: object

In [12]:
# ============================================================
# SECTION 5.3 - Missing Values
# ============================================================

missing_forenames = (
    forenames
    .isna()
    .sum()
    .to_frame("Missing Values")
)

missing_surnames = (
    surnames
    .isna()
    .sum()
    .to_frame("Missing Values")
)

print("FORNAMES")
display(missing_forenames)

print("SURNAMES")
display(missing_surnames)

FORNAMES


,Missing Values
FORENAME_UPPER,2
FORENAME_PROPER,2
FORENAME_MATCH,181175
EDIT_DISTANCE,0
JARO_WINKLER,0
FORENAME_FLAG,0
AMOUNT,0
GENDER,0
GENDER_COUNT,0


SURNAMES


,Missing Values
SURNAME_UPPER,3
SURNAME_PROPER,2
SURNAME_MATCH,2649591
EDIT_DISTANCE,0
JARO_WINKLER,0
SURNAME_FLAG,0
AMOUNT,0


In [13]:
# ============================================================
# SECTION 5.4 - Duplicate Records
# ============================================================

duplicates = pd.DataFrame({

    "Dataset":[
        "Forenames",
        "Surnames"
    ],

    "Duplicate Rows":[
        forenames.duplicated().sum(),
        surnames.duplicated().sum()
    ]

})

duplicates

,Dataset,Duplicate Rows
0,Forenames,0
1,Surnames,0


In [14]:
# ==========================================================
# SECTION 5.5 - Memory Usage
# ==========================================================

import pandas as pd

memory_usage = pd.DataFrame({
    "Dataset": ["Forenames", "Surnames"],
    "Rows": [forenames.shape[0], surnames.shape[0]],
    "Columns": [forenames.shape[1], surnames.shape[1]],
    "Memory (MB)": [
        round(forenames.memory_usage(deep=True).sum() / (1024 * 1024), 2),
        round(surnames.memory_usage(deep=True).sum() / (1024 * 1024), 2)
    ]
})

print("\nMemory Usage Summary")
display(memory_usage)


Memory Usage Summary


,Dataset,Rows,Columns,Memory (MB)
0,Forenames,204108,9,55.47
1,Surnames,4153336,7,911.06


In [15]:
# ==========================================================
# SECTION 5.6 - Descriptive Statistics
# ==========================================================

print("=" * 60)
print("FORENAME DESCRIPTIVE STATISTICS")
print("=" * 60)

display(forenames.describe(include="all"))

print("=" * 60)
print("SURNAME DESCRIPTIVE STATISTICS")
print("=" * 60)

display(surnames.describe(include="all"))

FORENAME DESCRIPTIVE STATISTICS


,FORENAME_UPPER,FORENAME_PROPER,FORENAME_MATCH,EDIT_DISTANCE,JARO_WINKLER,FORENAME_FLAG,AMOUNT,GENDER,GENDER_COUNT
count,204106,204106,22933,204108.000000,204108.000000,204108,2.041080e+05,204108,204108.000000
unique,204106,184488,4730,NaN,NaN,4,NaN,3,NaN
top,LISA-RAY,Margaret,MARGARET,NaN,NaN,G,NaN,U,NaN
freq,1,113,128,NaN,NaN,114027,NaN,84295,NaN
mean,NaN,NaN,NaN,0.172757,0.995190,NaN,1.268956e+04,NaN,-0.289121
std,NaN,NaN,NaN,0.528017,0.018306,NaN,3.290501e+05,NaN,1.580991
min,NaN,NaN,NaN,0.000000,0.000000,NaN,1.000000e+00,NaN,-5.000000
25%,NaN,NaN,NaN,0.000000,1.000000,NaN,1.300000e+01,NaN,-1.000000
50%,NaN,NaN,NaN,0.000000,1.000000,NaN,9.200000e+01,NaN,0.000000
75%,NaN,NaN,NaN,0.000000,1.000000,NaN,6.342500e+02,NaN,0.000000


SURNAME DESCRIPTIVE STATISTICS


,SURNAME_UPPER,SURNAME_PROPER,SURNAME_MATCH,EDIT_DISTANCE,JARO_WINKLER,SURNAME_FLAG,AMOUNT
count,4153333,4153334,1503745,4.153336e+06,4.153336e+06,4153336,4.153336e+06
unique,4153333,4151818,124959,NaN,NaN,4,NaN
top,BARRERA-DURAN,O'Brien,ROBINSON,NaN,NaN,U,NaN
freq,1,10,138,NaN,NaN,1991250,NaN
mean,NaN,NaN,NaN,3.805141e-01,9.794230e-01,NaN,8.611103e+02
std,NaN,NaN,NaN,5.242759e-01,3.775510e-02,NaN,4.716892e+04
min,NaN,NaN,NaN,0.000000e+00,0.000000e+00,NaN,1.000000e+00
25%,NaN,NaN,NaN,0.000000e+00,9.629630e-01,NaN,8.000000e+00
50%,NaN,NaN,NaN,0.000000e+00,1.000000e+00,NaN,2.900000e+01
75%,NaN,NaN,NaN,1.000000e+00,1.000000e+00,NaN,7.500000e+01


In [16]:
# ==========================================================
# SECTION 5.7 - Name Length Distribution
# ==========================================================

forename_lengths = (
    forenames["FORENAME_PROPER"]
    .fillna("")
    .astype(str)
    .str.len()
)

surname_lengths = (
    surnames["SURNAME_PROPER"]
    .fillna("")
    .astype(str)
    .str.len()
)

print("=" * 60)
print("FORENAME LENGTH DISTRIBUTION")
print("=" * 60)

display(
    forename_lengths
    .value_counts()
    .sort_index()
    .to_frame("Count")
)

print("=" * 60)
print("SURNAME LENGTH DISTRIBUTION")
print("=" * 60)

display(
    surname_lengths
    .value_counts()
    .sort_index()
    .to_frame("Count")
)

FORENAME LENGTH DISTRIBUTION


,Count
FORENAME_PROPER,
0,2
2,26
3,2828
4,7609
5,15693
6,19359
7,15582
8,12753
9,13880


SURNAME LENGTH DISTRIBUTION


,Count
SURNAME_PROPER,
0,2
1,26
2,662
3,7968
4,59595
5,266070
6,586453
7,750345
8,690338


In [17]:
# ==========================================================
# SECTION 5.8 - Character Composition
# ==========================================================

import re

def character_composition(name):

    name = str(name)

    if re.fullmatch(r"[A-Za-z]+", name):
        return "Alphabetic"

    elif re.search(r"\d", name):
        return "Contains Numbers"

    elif re.search(r"[^A-Za-z\s'-]", name):
        return "Special Characters"

    else:
        return "Mixed"

print("=" * 60)
print("FORENAME CHARACTER COMPOSITION")
print("=" * 60)

display(
    forenames["FORENAME_PROPER"]
    .fillna("")
    .apply(character_composition)
    .value_counts()
    .to_frame("Count")
)

print("=" * 60)
print("SURNAME CHARACTER COMPOSITION")
print("=" * 60)

display(
    surnames["SURNAME_PROPER"]
    .fillna("")
    .apply(character_composition)
    .value_counts()
    .to_frame("Count")
)

FORENAME CHARACTER COMPOSITION


,Count
FORENAME_PROPER,
Mixed,128089
Alphabetic,74427
Special Characters,1559
Contains Numbers,33


SURNAME CHARACTER COMPOSITION


,Count
SURNAME_PROPER,
Alphabetic,3441118
Mixed,704858
Special Characters,5663
Contains Numbers,1697


In [18]:
# ============================================================
# SECTION 5.8 - Statistical Summary
# ============================================================

print("FORNAMES")

display(forenames.describe(include="all"))

print("\nSURNAMES")

display(surnames.describe(include="all"))

FORNAMES


,FORENAME_UPPER,FORENAME_PROPER,FORENAME_MATCH,EDIT_DISTANCE,JARO_WINKLER,FORENAME_FLAG,AMOUNT,GENDER,GENDER_COUNT
count,204106,204106,22933,204108.000000,204108.000000,204108,2.041080e+05,204108,204108.000000
unique,204106,184488,4730,NaN,NaN,4,NaN,3,NaN
top,LISA-RAY,Margaret,MARGARET,NaN,NaN,G,NaN,U,NaN
freq,1,113,128,NaN,NaN,114027,NaN,84295,NaN
mean,NaN,NaN,NaN,0.172757,0.995190,NaN,1.268956e+04,NaN,-0.289121
std,NaN,NaN,NaN,0.528017,0.018306,NaN,3.290501e+05,NaN,1.580991
min,NaN,NaN,NaN,0.000000,0.000000,NaN,1.000000e+00,NaN,-5.000000
25%,NaN,NaN,NaN,0.000000,1.000000,NaN,1.300000e+01,NaN,-1.000000
50%,NaN,NaN,NaN,0.000000,1.000000,NaN,9.200000e+01,NaN,0.000000
75%,NaN,NaN,NaN,0.000000,1.000000,NaN,6.342500e+02,NaN,0.000000



SURNAMES


,SURNAME_UPPER,SURNAME_PROPER,SURNAME_MATCH,EDIT_DISTANCE,JARO_WINKLER,SURNAME_FLAG,AMOUNT
count,4153333,4153334,1503745,4.153336e+06,4.153336e+06,4153336,4.153336e+06
unique,4153333,4151818,124959,NaN,NaN,4,NaN
top,BARRERA-DURAN,O'Brien,ROBINSON,NaN,NaN,U,NaN
freq,1,10,138,NaN,NaN,1991250,NaN
mean,NaN,NaN,NaN,3.805141e-01,9.794230e-01,NaN,8.611103e+02
std,NaN,NaN,NaN,5.242759e-01,3.775510e-02,NaN,4.716892e+04
min,NaN,NaN,NaN,0.000000e+00,0.000000e+00,NaN,1.000000e+00
25%,NaN,NaN,NaN,0.000000e+00,9.629630e-01,NaN,8.000000e+00
50%,NaN,NaN,NaN,0.000000e+00,1.000000e+00,NaN,2.900000e+01
75%,NaN,NaN,NaN,1.000000e+00,1.000000e+00,NaN,7.500000e+01


# SECTION 6 – Business Rule Analysis

## Purpose

The PDSA datasets contain classification flags that describe the operational status of each name.

Before performing any data cleansing or matching, it is important to understand how these records are distributed across the different flag categories.

This analysis provides insight into the proportion of names that are already accepted, require matching, remain unclassified, or have been rejected. These classifications guide the subsequent cleaning, normalisation and matching stages of the workflow.

**Note:** The interpretation of the flag values is based on analysis of the dataset structure and behaviour rather than official PDSA documentation.

In [19]:
# ============================================================
# SECTION 6.1 - Forename Flag Distribution
# ============================================================

forename_flags = (
    forenames["FORENAME_FLAG"]
    .value_counts()
    .rename_axis("Flag")
    .reset_index(name="Count")
)

forename_flags["Percentage"] = (
    forename_flags["Count"] /
    len(forenames) * 100
).round(2)

display(forename_flags)

,Flag,Count,Percentage
0,G,114027,55.87
1,U,66878,32.77
2,A,22933,11.24
3,R,270,0.13


### 6.2 Inspect Representative Records

To better understand the operational meaning of each flag, representative records are extracted from each category.

Rather than relying solely on frequency counts, this inspection helps determine how PDSA uses each flag in practice and how the flags influence later matching decisions.

In [20]:
# ============================================================
# SECTION 6.2 - Sample Records by Flag
# ============================================================

for flag in sorted(forenames["FORENAME_FLAG"].dropna().unique()):

    print("=" * 80)
    print(f"FORENAME FLAG = {flag}")
    print("=" * 80)

    display(
        forenames.loc[
            forenames["FORENAME_FLAG"] == flag,
            [
                "FORENAME_PROPER",
                "FORENAME_MATCH",
                "FORENAME_FLAG"
            ]
        ].head(10)
    )

FORENAME FLAG = A


,FORENAME_PROPER,FORENAME_MATCH,FORENAME_FLAG
541,Michael,MICHAEL,A
751,Kimberley,KIMBERLEY,A
797,David,DAVID,A
836,Shelley,SHELLEY,A
867,Michael,MICHAEL,A
887,Darren,DARREN,A
908,Fredrick,FREDERICK,A
1026,Lindsey,LINDSEY,A
1052,Patricia,PATRICIA,A
1057,Julie,JULIE,A


FORENAME FLAG = G


,FORENAME_PROPER,FORENAME_MATCH,FORENAME_FLAG
0,John,NaN,G
1,David,NaN,G
2,James,NaN,G
3,Michael,NaN,G
4,Paul,NaN,G
5,Andrew,NaN,G
6,Robert,NaN,G
7,Margaret,NaN,G
8,Sarah,NaN,G
9,Peter,NaN,G


FORENAME FLAG = R


,FORENAME_PROPER,FORENAME_MATCH,FORENAME_FLAG
480,The,NaN,R
715,Miss,NaN,R
773,Mrs,NaN,R
1014,Exec,NaN,R
1386,Jones,NaN,R
1822,Junior,NaN,R
1885,Prince,NaN,R
2455,And,NaN,R
2841,King,NaN,R
2951,Test,NaN,R


FORENAME FLAG = U


,FORENAME_PROPER,FORENAME_MATCH,FORENAME_FLAG
913,Przemyslaw,NaN,U
982,Mateusz,NaN,U
1039,Arkadiusz,NaN,U
1060,Dawid,NaN,U
1119,Bartosz,NaN,U
1131,Patrycja,NaN,U
1144,Radoslaw,NaN,U
1251,Miroslaw,NaN,U
1266,Kinga,NaN,U
1322,Alexandru,NaN,U


In [21]:
# ============================================================
# SECTION 6.3 - Partitioning Dataset by Business Rules
# ============================================================

canonical_names = (
    forenames[forenames["FORENAME_FLAG"] == "G"]
    .copy()
)

accepted_variants = (
    forenames[forenames["FORENAME_FLAG"] == "A"]
    .copy()
)

unknown_names = (
    forenames[forenames["FORENAME_FLAG"] == "U"]
    .copy()
)

rejected_names = (
    forenames[forenames["FORENAME_FLAG"] == "R"]
    .copy()
)

summary = pd.DataFrame({

    "Dataset":[
        "Canonical Names",
        "Accepted Variants",
        "Unknown Names",
        "Rejected Names"
    ],

    "Records":[
        len(canonical_names),
        len(accepted_variants),
        len(unknown_names),
        len(rejected_names)
    ]

})

summary

,Dataset,Records
0,Canonical Names,114027
1,Accepted Variants,22933
2,Unknown Names,66878
3,Rejected Names,270


In [22]:
# SECTION 7 – Data Quality Assessment

# Purpose

# This section evaluates the quality of the names that require further processing.

# The assessment identifies common data quality issues that may reduce the effectiveness of name matching algorithms, including:

# - Missing names
# - Blank values
# - Leading/trailing spaces
# - Multiple spaces
# - Hyphenated names
# - Multi-word names
# - Apostrophes
# - Non-ASCII characters

# The results determine which cleaning operations are required before name normalisation and matching.

In [23]:
# ============================================================
# SECTION 7.1 - Data Quality Assessment
# ============================================================

quality_summary = pd.DataFrame({

    "Dataset":[
        "Canonical",
        "Accepted Variants",
        "Unknown",
        "Rejected"
    ],

    "Records":[
        len(canonical_names),
        len(accepted_variants),
        len(unknown_names),
        len(rejected_names)
    ],

    "Missing Names":[
        canonical_names["FORENAME_PROPER"].isna().sum(),
        accepted_variants["FORENAME_PROPER"].isna().sum(),
        unknown_names["FORENAME_PROPER"].isna().sum(),
        rejected_names["FORENAME_PROPER"].isna().sum()
    ],

    "Blank Names":[
        canonical_names["FORENAME_PROPER"].eq("").sum(),
        accepted_variants["FORENAME_PROPER"].eq("").sum(),
        unknown_names["FORENAME_PROPER"].eq("").sum(),
        rejected_names["FORENAME_PROPER"].eq("").sum()
    ]

})

quality_summary

,Dataset,Records,Missing Names,Blank Names
0,Canonical,114027,0,0
1,Accepted Variants,22933,0,0
2,Unknown,66878,2,0
3,Rejected,270,0,0


In [24]:
# ============================================================
# Leading / Trailing Spaces
# ============================================================

space_summary = pd.DataFrame({

    "Dataset":[
        "Canonical",
        "Accepted Variants",
        "Unknown"
    ],

    "Leading/Trailing Spaces":[

        canonical_names["FORENAME_PROPER"]
        .str.startswith(" ")
        .fillna(False)
        .sum()

        +

        canonical_names["FORENAME_PROPER"]
        .str.endswith(" ")
        .fillna(False)
        .sum(),


        accepted_variants["FORENAME_PROPER"]
        .str.startswith(" ")
        .fillna(False)
        .sum()

        +

        accepted_variants["FORENAME_PROPER"]
        .str.endswith(" ")
        .fillna(False)
        .sum(),


        unknown_names["FORENAME_PROPER"]
        .str.startswith(" ")
        .fillna(False)
        .sum()

        +

        unknown_names["FORENAME_PROPER"]
        .str.endswith(" ")
        .fillna(False)
        .sum()

    ]

})

space_summary

C:\Users\omoto\AppData\Local\Temp\ipykernel_41096\520048199.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
C:\Users\omoto\AppData\Local\Temp\ipykernel_41096\520048199.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,Dataset,Leading/Trailing Spaces
0,Canonical,0
1,Accepted Variants,0
2,Unknown,0


In [25]:
# ============================================================
# Hyphenated Names
# ============================================================

hyphen_summary = pd.DataFrame({

    "Dataset":[
        "Canonical",
        "Accepted Variants",
        "Unknown"
    ],

    "Hyphenated Names":[

        canonical_names["FORENAME_PROPER"]
        .str.contains("-", regex=False, na=False)
        .sum(),

        accepted_variants["FORENAME_PROPER"]
        .str.contains("-", regex=False, na=False)
        .sum(),

        unknown_names["FORENAME_PROPER"]
        .str.contains("-", regex=False, na=False)
        .sum()

    ]

})

hyphen_summary

,Dataset,Hyphenated Names
0,Canonical,67566
1,Accepted Variants,704
2,Unknown,11445


In [26]:
# ============================================================
# Multi-word Names
# ============================================================

multiword_summary = pd.DataFrame({

    "Dataset":[
        "Canonical",
        "Accepted Variants",
        "Unknown"
    ],

    "Multi-word Names":[

        canonical_names["FORENAME_PROPER"]
        .str.contains(" ", regex=False, na=False)
        .sum(),

        accepted_variants["FORENAME_PROPER"]
        .str.contains(" ", regex=False, na=False)
        .sum(),

        unknown_names["FORENAME_PROPER"]
        .str.contains(" ", regex=False, na=False)
        .sum()

    ]

})

multiword_summary

,Dataset,Multi-word Names
0,Canonical,43237
1,Accepted Variants,1957
2,Unknown,4223


In [27]:
# ============================================================
# Apostrophes
# ============================================================

apostrophe_summary = pd.DataFrame({

    "Dataset":[
        "Canonical",
        "Accepted Variants",
        "Unknown"
    ],

    "Names with Apostrophe":[

        canonical_names["FORENAME_PROPER"]
        .str.contains("'", regex=False, na=False)
        .sum(),

        accepted_variants["FORENAME_PROPER"]
        .str.contains("'", regex=False, na=False)
        .sum(),

        unknown_names["FORENAME_PROPER"]
        .str.contains("'", regex=False, na=False)
        .sum()

    ]

})

apostrophe_summary

,Dataset,Names with Apostrophe
0,Canonical,11
1,Accepted Variants,6
2,Unknown,39


In [28]:
# ============================================================
# Non-ASCII Characters
# ============================================================

def contains_non_ascii(text):

    if pd.isna(text):
        return False

    return any(ord(c) > 127 for c in str(text))


ascii_summary = pd.DataFrame({

    "Dataset":[
        "Canonical",
        "Accepted Variants",
        "Unknown"
    ],

    "Non-ASCII Names":[

        canonical_names["FORENAME_PROPER"]
        .apply(contains_non_ascii)
        .sum(),

        accepted_variants["FORENAME_PROPER"]
        .apply(contains_non_ascii)
        .sum(),

        unknown_names["FORENAME_PROPER"]
        .apply(contains_non_ascii)
        .sum()

    ]

})

ascii_summary

,Dataset,Non-ASCII Names
0,Canonical,0
1,Accepted Variants,0
2,Unknown,4


# SECTION 8 – Data Cleaning

## Purpose

This section prepares the names for normalisation and matching by applying consistent cleaning rules.

The cleaning process:

- Removes leading and trailing spaces.
- Removes multiple consecutive spaces.
- Converts blank strings to missing values.
- Preserves the original datasets by creating cleaned copies.

The cleaned datasets form the input to the name normalisation stage.

In [29]:
# ============================================================
# SECTION 8.1 - Createing Working Copies
# ============================================================

canonical_clean = canonical_names.copy()

accepted_clean = accepted_variants.copy()

unknown_clean = unknown_names.copy()

print("Working copies created successfully.")

Working copies created successfully.


In [30]:
# ============================================================
# SECTION 8.2 - Name Cleaning Function
# ============================================================

def clean_name(name):

    if pd.isna(name):
        return np.nan

    name = str(name)

    # Remove leading/trailing spaces
    name = name.strip()

    # Replace multiple spaces with a single space
    name = re.sub(r"\s+", " ", name)

    # Convert empty strings to missing values
    if name == "":
        return np.nan

    return name

In [31]:
# Importing the required modules
import re
import numpy as np  # Added NumPy import to fix the NameError
import pandas as pd  # Also adding pandas as it's likely needed for DataFrame operations

# ============================================================
# SECTION 8.3 - Applying Cleaning
# ============================================================

canonical_clean["CLEAN_NAME"] = canonical_clean["FORENAME_PROPER"].apply(clean_name)

accepted_clean["CLEAN_NAME"] = accepted_clean["FORENAME_PROPER"].apply(clean_name)

unknown_clean["CLEAN_NAME"] = unknown_clean["FORENAME_PROPER"].apply(clean_name)

print("Cleaning completed successfully.")

Cleaning completed successfully.


In [32]:
# ============================================================
# SECTION 8.4 - Cleaning Verification
# ============================================================

cleaning_summary = pd.DataFrame({

    "Dataset": [
        "Canonical",
        "Accepted Variants",
        "Unknown"
    ],

    "Original Missing": [

        canonical_names["FORENAME_PROPER"].isna().sum(),

        accepted_variants["FORENAME_PROPER"].isna().sum(),

        unknown_names["FORENAME_PROPER"].isna().sum()

    ],

    "Clean Missing": [

        canonical_clean["CLEAN_NAME"].isna().sum(),

        accepted_clean["CLEAN_NAME"].isna().sum(),

        unknown_clean["CLEAN_NAME"].isna().sum()

    ]

})

cleaning_summary

,Dataset,Original Missing,Clean Missing
0,Canonical,0,0
1,Accepted Variants,0,0
2,Unknown,2,2


# SECTION 9 – Name Normalisation

## Purpose

This section standardises the cleaned names into a consistent format suitable for matching.

The normalisation process:

- Converts names to uppercase.
- Removes accented characters.
- Removes leading/trailing whitespace (already completed during cleaning).
- Produces a standard representation for all subsequent matching algorithms.

The normalised names form the basis for rule-based matching, variant detection and fuzzy matching.

In [33]:
# ============================================================
# SECTION 9.1 - Name Normalisation Function
# ============================================================

def normalise_name(name):

    if pd.isna(name):
        return np.nan

    # Remove accents
    name = unidecode(str(name))

    # Convert to uppercase
    name = name.upper()

    return name

In [34]:
# Import the required library
from unidecode import unidecode

# ============================================================
# SECTION 9.2 - Applying Normalisation
# ============================================================

canonical_clean["NORMALISED_NAME"] = (
    canonical_clean["CLEAN_NAME"]
    .apply(normalise_name)
)

accepted_clean["NORMALISED_NAME"] = (
    accepted_clean["CLEAN_NAME"]
    .apply(normalise_name)
)

unknown_clean["NORMALISED_NAME"] = (
    unknown_clean["CLEAN_NAME"]
    .apply(normalise_name)
)

print("Name normalisation completed successfully.")

Name normalisation completed successfully.


In [35]:
# ============================================================
# SECTION 9.3 - Verification
# ============================================================

display(

    unknown_clean[
        [
            "FORENAME_PROPER",
            "CLEAN_NAME",
            "NORMALISED_NAME"
        ]
    ].head(20)

)

,FORENAME_PROPER,CLEAN_NAME,NORMALISED_NAME
913,Przemyslaw,Przemyslaw,PRZEMYSLAW
982,Mateusz,Mateusz,MATEUSZ
1039,Arkadiusz,Arkadiusz,ARKADIUSZ
1060,Dawid,Dawid,DAWID
1119,Bartosz,Bartosz,BARTOSZ
1131,Patrycja,Patrycja,PATRYCJA
1144,Radoslaw,Radoslaw,RADOSLAW
1251,Miroslaw,Miroslaw,MIROSLAW
1266,Kinga,Kinga,KINGA
1322,Alexandru,Alexandru,ALEXANDRU


In [36]:
# ==========================================================
# CREATE STANDARD WORKING COLUMNS
# ==========================================================

from unidecode import unidecode

# ---------- Forenames ----------
df_forenames = forenames.copy()

df_forenames["Original_Name"] = df_forenames["FORENAME_PROPER"]

df_forenames["NORMALISED_NAME"] = (
    df_forenames["FORENAME_PROPER"]
        .astype(str)
        .str.strip()
        .str.upper()
        .apply(unidecode)
)

# ---------- Surnames ----------
df_surnames = surnames.copy()

df_surnames["Original_Name"] = df_surnames["SURNAME_PROPER"]

df_surnames["NORMALISED_NAME"] = (
    df_surnames["SURNAME_PROPER"]
        .astype(str)
        .str.strip()
        .str.upper()
        .apply(unidecode)
)

print(df_forenames[["Original_Name","NORMALISED_NAME"]].head())

print(df_surnames[["Original_Name","NORMALISED_NAME"]].head())

  Original_Name NORMALISED_NAME
0          John            JOHN
1         David           DAVID
2         James           JAMES
3       Michael         MICHAEL
4          Paul            PAUL
  Original_Name NORMALISED_NAME
0         Smith           SMITH
1         Jones           JONES
2      Williams        WILLIAMS
3         Brown           BROWN
4        Taylor          TAYLOR


#### SECTION 10 – Rule-Based Standardisation

## Purpose

Before applying approximate string matching, known name variants are resolved using deterministic business rules.

A variant dictionary is used to map common abbreviations, nicknames and alternative spellings to their canonical names.

This reduces unnecessary fuzzy matching and improves both speed and accuracy.

In [37]:
# ============================================================
# SECTION 10.1 - Loading Variant Dictionary
# ============================================================

# Define PROJECT_DIR - adjust this path to match your project structure
import pandas as pd
from pathlib import Path

# Option 1: If the CSV is in the same directory as your notebook
PROJECT_DIR = Path(".")

# Option 2: If you have a specific project directory structure
# PROJECT_DIR = Path("/path/to/your/project")  # Replace with your actual path

# Option 3: If the CSV is in a subdirectory called 'data'
# PROJECT_DIR = Path("data")

variant_dictionary = pd.read_csv(PROJECT_DIR / "NAME_VARIANTS.csv")

print(f"Variant dictionary loaded successfully.")

print(f"Records: {len(variant_dictionary):,}")

display(variant_dictionary.head(10))

Variant dictionary loaded successfully.
Records: 26,589


,ORIGINAL_NAME,CANONICAL_NAME,MATCH_TYPE,CONFIDENCE
0,Fredrick,FREDERICK,SPELLING_VARIANT,0.95
1,Niki,NIKKI,SPELLING_VARIANT,0.95
2,Florin,FLORIAN,SPELLING_VARIANT,0.95
3,Christoph,CHRISTOPHE,SPELLING_VARIANT,0.95
4,Krystian,KRYSTINA,SPELLING_VARIANT,0.95
5,Andreea,ANDREA,SPELLING_VARIANT,0.95
6,Em,EMMA,SPELLING_VARIANT,0.95
7,Viktorija,VIKTORIA,SPELLING_VARIANT,0.95
8,Georgi,GEORGIA,SPELLING_VARIANT,0.95
9,Martyna,MARTYN,SPELLING_VARIANT,0.95


In [38]:
variant_dictionary.columns.tolist()

['ORIGINAL_NAME', 'CANONICAL_NAME', 'MATCH_TYPE', 'CONFIDENCE']

In [39]:
# ==========================================================
# SECTION 10 - INVALID NAME CLASSIFICATION
# ==========================================================

import re
import pandas as pd

INVALID_WORDS = {
    "",
    "UNKNOWN",
    "UNK",
    "N/A",
    "NA",
    "NONE",
    "NULL",
    "NOT PROVIDED",
    "TEST",
    "TEMP",
    "DUMMY",
    "XXXXX",
    "XXXX",
    "XXX",
    "AAAA",
    "ZZZZ",
    "ABC",
    "QWERTY"
}

def classify_invalid_name(name):

    if pd.isna(name):
        return "Missing"

    name = str(name).strip().upper()

    if name in INVALID_WORDS:
        return "Placeholder"

    if len(name) < 2:
        return "Too Short"

    if re.fullmatch(r"\d+", name):
        return "Numeric"

    if re.fullmatch(r"(.)\1{3,}", name):
        return "Repeated Characters"

    if re.fullmatch(r"[^A-Z]+", name):
        return "Symbols"

    return "Valid"

In [40]:
# ---------- Forenames ----------

df_forenames["INVALID_REASON"] = (
    df_forenames["FORENAME_UPPER"]
    .apply(classify_invalid_name)
)

df_forenames["INVALID_NAME"] = (
    df_forenames["INVALID_REASON"] != "Valid"
)

# ---------- Surnames ----------

df_surnames["INVALID_REASON"] = (
    df_surnames["SURNAME_UPPER"]
    .apply(classify_invalid_name)
)

df_surnames["INVALID_NAME"] = (
    df_surnames["INVALID_REASON"] != "Valid"
)

In [41]:
print("\nForenames")
display(
    df_forenames["INVALID_REASON"].value_counts().to_frame()
)

print("\nSurnames")
display(
    df_surnames["INVALID_REASON"].value_counts().to_frame()
)


Forenames


,count
INVALID_REASON,
Valid,204081
Repeated Characters,19
Placeholder,6
Missing,2



Surnames


,count
INVALID_REASON,
Valid,4153176
Repeated Characters,119
Too Short,26
Placeholder,12
Missing,3


In [42]:
display(
    df_forenames.loc[
        df_forenames["INVALID_NAME"],
        [
            "FORENAME_PROPER",
            "INVALID_REASON"
        ]
    ].head(20)
)

display(
    df_surnames.loc[
        df_surnames["INVALID_NAME"],
        [
            "SURNAME_PROPER",
            "INVALID_REASON"
        ]
    ].head(20)
)

,FORENAME_PROPER,INVALID_REASON
2951,Test,Placeholder
4407,NaN,Placeholder
7283,Unknown,Placeholder
13372,NaN,Missing
25518,Abc,Placeholder
26204,Null,Missing
46224,Xxxx,Placeholder
56931,Jjjj,Repeated Characters
56960,Dddd,Repeated Characters
57797,Xxxxx,Placeholder


,SURNAME_PROPER,INVALID_REASON
4096,M,Too Short
4756,J,Too Short
4797,A,Too Short
6894,E,Too Short
6969,H,Too Short
8123,R,Too Short
8406,C,Too Short
8756,S,Too Short
8757,D,Too Short
9165,B,Too Short


In [43]:
clean_forenames = df_forenames[
    ~df_forenames["INVALID_NAME"]
].copy()

clean_surnames = df_surnames[
    ~df_surnames["INVALID_NAME"]
].copy()

print(f"Valid Forenames: {len(clean_forenames):,}")
print(f"Valid Surnames: {len(clean_surnames):,}")

Valid Forenames: 204,081
Valid Surnames: 4,153,176


In [44]:
clean_forenames
clean_surnames

,SURNAME_UPPER,SURNAME_PROPER,SURNAME_MATCH,EDIT_DISTANCE,JARO_WINKLER,SURNAME_FLAG,AMOUNT,Original_Name,NORMALISED_NAME,INVALID_REASON,INVALID_NAME
0,SMITH,Smith,NaN,0,1.0,G,39891647,Smith,SMITH,Valid,False
1,JONES,Jones,NaN,0,1.0,G,29317272,Jones,JONES,Valid,False
2,WILLIAMS,Williams,NaN,0,1.0,G,20497885,Williams,WILLIAMS,Valid,False
3,BROWN,Brown,NaN,0,1.0,G,18821086,Brown,BROWN,Valid,False
4,TAYLOR,Taylor,NaN,0,1.0,G,18390861,Taylor,TAYLOR,Valid,False
...,...,...,...,...,...,...,...,...,...,...,...
4153331,KRAL-BREWER,Kral-Brewer,NaN,0,1.0,G,1,Kral-Brewer,KRAL-BREWER,Valid,False
4153332,PAULIN-CANVIN,Paulin-Canvin,NaN,0,1.0,G,1,Paulin-Canvin,PAULIN-CANVIN,Valid,False
4153333,KERSHAW-CARSON,Kershaw-Carson,NaN,0,1.0,G,1,Kershaw-Carson,KERSHAW-CARSON,Valid,False
4153334,MACINTYRE-RICHINGS,Macintyre-Richings,NaN,0,1.0,G,1,Macintyre-Richings,MACINTYRE-RICHINGS,Valid,False


In [45]:
# ==========================================================
# Script Detection Function
# ==========================================================

import unicodedata

def detect_script(text):
    """
    Detect the writing system used in a name.
    """

    if text is None:
        return "Unknown"

    text = str(text).strip()

    if text == "":
        return "Unknown"

    for char in text:

        try:
            name = unicodedata.name(char)

            if "LATIN" in name:
                return "Latin"

            elif "CYRILLIC" in name:
                return "Cyrillic"

            elif "ARABIC" in name:
                return "Arabic"

            elif "GREEK" in name:
                return "Greek"

            elif "HEBREW" in name:
                return "Hebrew"

            elif "CJK" in name:
                return "Chinese"

            elif "HIRAGANA" in name:
                return "Japanese"

            elif "KATAKANA" in name:
                return "Japanese"

            elif "HANGUL" in name:
                return "Korean"

        except ValueError:
            continue

    return "Unknown"

In [46]:
clean_forenames["SCRIPT"] = (
    clean_forenames["Original_Name"]
    .apply(detect_script)
)

clean_surnames["SCRIPT"] = (
    clean_surnames["Original_Name"]
    .apply(detect_script)
)

In [47]:
# ==========================================================
# Detect Script from Unicode Characters
# ==========================================================

def detect_script(name):

    if pd.isna(name):
        return "Unknown"

    name = str(name).strip()

    if len(name) == 0:
        return "Unknown"

    for char in name:

        try:
            unicode_name = unicodedata.name(char)

        except ValueError:
            continue

        if "ARABIC" in unicode_name:
            return "Arabic"

        elif "CYRILLIC" in unicode_name:
            return "Cyrillic"

        elif "GREEK" in unicode_name:
            return "Greek"

        elif "HEBREW" in unicode_name:
            return "Hebrew"

        elif "HIRAGANA" in unicode_name:
            return "Japanese"

        elif "KATAKANA" in unicode_name:
            return "Japanese"

        elif "CJK" in unicode_name:
            return "Chinese"

        elif "HANGUL" in unicode_name:
            return "Korean"

        elif "LATIN" in unicode_name:
            continue

    return "Latin"

In [48]:
# ==========================================================
# Script Summary
# ==========================================================

print("\nForenames Script Distribution")

display(
    clean_forenames["SCRIPT"]
    .value_counts()
    .to_frame("Count")
)

print("\nSurnames Script Distribution")

display(
    clean_surnames["SCRIPT"]
    .value_counts()
    .to_frame("Count")
)


Forenames Script Distribution


,Count
SCRIPT,
Latin,204081



Surnames Script Distribution


,Count
SCRIPT,
Latin,4153176


In [49]:
# ==========================================================
# Non-Latin Examples
# ==========================================================

print("\nNon-Latin Forenames")

display(
    clean_forenames[
        clean_forenames["SCRIPT"] != "Latin"
    ][
        [
            "FORENAME_PROPER",
            "SCRIPT"
        ]
    ].head(20)
)

print("\nNon-Latin Surnames")

display(
    clean_surnames[
        clean_surnames["SCRIPT"] != "Latin"
    ][
        [
            "SURNAME_PROPER",
            "SCRIPT"
        ]
    ].head(20)
)


Non-Latin Forenames


,FORENAME_PROPER,SCRIPT



Non-Latin Surnames


,SURNAME_PROPER,SCRIPT


In [50]:
variant_dictionary.head(10)

,ORIGINAL_NAME,CANONICAL_NAME,MATCH_TYPE,CONFIDENCE
0,Fredrick,FREDERICK,SPELLING_VARIANT,0.95
1,Niki,NIKKI,SPELLING_VARIANT,0.95
2,Florin,FLORIAN,SPELLING_VARIANT,0.95
3,Christoph,CHRISTOPHE,SPELLING_VARIANT,0.95
4,Krystian,KRYSTINA,SPELLING_VARIANT,0.95
5,Andreea,ANDREA,SPELLING_VARIANT,0.95
6,Em,EMMA,SPELLING_VARIANT,0.95
7,Viktorija,VIKTORIA,SPELLING_VARIANT,0.95
8,Georgi,GEORGIA,SPELLING_VARIANT,0.95
9,Martyna,MARTYN,SPELLING_VARIANT,0.95


In [51]:
# ==========================================================
# - DATA QUALITY DASHBOARD
# ==========================================================

import pandas as pd

forename_dashboard = {

    "Dataset": "Forenames",

    "Total Records": len(df_forenames),

    "Valid Records": len(clean_forenames),

    "Invalid Records": df_forenames["INVALID_NAME"].sum(),

    "Missing Values":
        df_forenames["FORENAME_PROPER"].isna().sum(),

    "Duplicate Names":
        df_forenames["FORENAME_UPPER"].duplicated().sum(),

    "Unique Names":
        df_forenames["FORENAME_UPPER"].nunique()

}

surname_dashboard = {

    "Dataset": "Surnames",

    "Total Records": len(df_surnames),

    "Valid Records": len(clean_surnames),

    "Invalid Records": df_surnames["INVALID_NAME"].sum(),

    "Missing Values":
        df_surnames["SURNAME_PROPER"].isna().sum(),

    "Duplicate Names":
        df_surnames["SURNAME_UPPER"].duplicated().sum(),

    "Unique Names":
        df_surnames["SURNAME_UPPER"].nunique()

}

quality_dashboard = pd.DataFrame(
    [forename_dashboard, surname_dashboard]
)

quality_dashboard

,Dataset,Total Records,Valid Records,Invalid Records,Missing Values,Duplicate Names,Unique Names
0,Forenames,204108,204081,27,2,1,204106
1,Surnames,4153336,4153176,160,2,2,4153333


In [52]:
quality_dashboard["Invalid %"] = (
    quality_dashboard["Invalid Records"]
    / quality_dashboard["Total Records"]
    * 100
).round(3)

quality_dashboard["Duplicate %"] = (
    quality_dashboard["Duplicate Names"]
    / quality_dashboard["Total Records"]
    * 100
).round(3)

quality_dashboard["Missing %"] = (
    quality_dashboard["Missing Values"]
    / quality_dashboard["Total Records"]
    * 100
).round(3)

quality_dashboard

,Dataset,Total Records,Valid Records,Invalid Records,Missing Values,Duplicate Names,Unique Names,Invalid %,Duplicate %,Missing %
0,Forenames,204108,204081,27,2,1,204106,0.013,0.0,0.001
1,Surnames,4153336,4153176,160,2,2,4153333,0.004,0.0,0.000


In [53]:
quality_dashboard.to_csv(
    "Data_Quality_Dashboard.csv",
    index=False
)

print("✓ Data Quality Dashboard exported.")

✓ Data Quality Dashboard exported.


### 14.2 Apply Rule-Based Standardisation

Known spelling variants are matched against the rule-based dictionary before approximate string matching is performed.

This deterministic step ensures that previously identified variants are resolved immediately without requiring fuzzy matching, thereby improving both efficiency and accuracy.

In [54]:
# ============================================================
# SECTION 10.2 - Applying Rule-Based Standardisation
# ============================================================

# Create lookup dictionary
rule_lookup = dict(
    zip(
        variant_dictionary["ORIGINAL_NAME"].str.upper(),
        variant_dictionary["CANONICAL_NAME"].str.upper()
    )
)

# Apply lookup to unknown names
unknown_clean["RULE_MATCH"] = (
    unknown_clean["NORMALISED_NAME"]
    .map(rule_lookup)
)

# Final name after rule-based matching
unknown_clean["STANDARDISED_NAME"] = (
    unknown_clean["RULE_MATCH"]
    .fillna(unknown_clean["NORMALISED_NAME"])
)

print("Rule-based matching completed.")

Rule-based matching completed.


In [55]:
# ============================================================
# Verification
# ============================================================

unknown_clean[
    unknown_clean["RULE_MATCH"].notna()
][
    [
        "FORENAME_PROPER",
        "NORMALISED_NAME",
        "RULE_MATCH",
        "STANDARDISED_NAME"
    ]
].head(20)

,FORENAME_PROPER,NORMALISED_NAME,RULE_MATCH,STANDARDISED_NAME
1131,Patrycja,PATRYCJA,PATRICIA,PATRICIA
1251,Miroslaw,MIROSLAW,MIROSLAV,MIROSLAV
1443,Mohd,MOHD,MOHAMMED,MOHAMMED
7487,Jean-Baptiste,JEAN-BAPTISTE,BAPTISTE,BAPTISTE
8941,Marie-France,MARIE-FRANCE,FRANCE,FRANCE
10686,Sammie-Jo,SAMMIE-JO,JO,JO
12406,Le-Anne,LE-ANNE,ANNE,ANNE
14041,Marie-Ange,MARIE-ANGE,ANGE,ANGE
14313,Mei-Ling,MEI-LING,LING,LING
14975,Sammi-Jo,SAMMI-JO,JO,JO


In [56]:
unknown_clean["RULE_MATCH"].notna().sum()

np.int64(11448)

In [57]:
# ============================================================
# SECTION 10.3 - Protecting Compound Names
# ============================================================

# Restoring compound names (hyphenated names) to their original normalised form
compound_mask = unknown_clean["NORMALISED_NAME"].str.contains("-", regex=False, na=False)

unknown_clean.loc[compound_mask, "RULE_MATCH"] = np.nan

unknown_clean.loc[compound_mask, "STANDARDISED_NAME"] = (
    unknown_clean.loc[compound_mask, "NORMALISED_NAME"]
)

print(f"Protected {compound_mask.sum():,} compound names from incorrect rule matching.")

Protected 11,445 compound names from incorrect rule matching.


In [58]:
# Verification

unknown_clean[
    unknown_clean["FORENAME_PROPER"].isin([
        "Jean-Baptiste",
        "Marie-France",
        "Sammie-Jo",
        "Abdul-Rahman",
        "Mei-Ling"
    ])
][
    [
        "FORENAME_PROPER",
        "RULE_MATCH",
        "STANDARDISED_NAME"
    ]
]

,FORENAME_PROPER,RULE_MATCH,STANDARDISED_NAME
7487,Jean-Baptiste,NaN,JEAN-BAPTISTE
8941,Marie-France,NaN,MARIE-FRANCE
10686,Sammie-Jo,NaN,SAMMIE-JO
14313,Mei-Ling,NaN,MEI-LING
20265,Abdul-Rahman,NaN,ABDUL-RAHMAN


# SECTION 11 – Compound Name Enhancement

## Purpose

Compound names require specialised handling because they consist of multiple name components connected by a hyphen.

Rather than allowing the rule-based matching process to incorrectly map a compound name to one of its components, a dedicated compound enhancement matrix is used to preserve and identify these names before approximate matching.

This stage enhances matching accuracy by ensuring that compound names are processed correctly during candidate generation.

In [59]:
# ============================================================
# SECTION 11.1 - Loading Compound Enhancement Matrix
# ============================================================

compound_enhancement_matrix = pd.read_csv(
    PROJECT_DIR / "compound_enhancement_matrix.csv"
)

print("Compound enhancement matrix loaded successfully.")
print(f"Rows: {len(compound_enhancement_matrix):,}")

display(compound_enhancement_matrix.head(10))

Compound enhancement matrix loaded successfully.
Rows: 23,275


,ORIGINAL_NAME,CANONICAL_NAME,MATCH_TYPE,CONFIDENCE
0,Jean-Baptiste,Jean,COMPOUND_DECOMPOSITION,0.95
1,Jean-Baptiste,Baptiste,COMPOUND_DECOMPOSITION,0.95
2,Marie-France,Marie,COMPOUND_DECOMPOSITION,0.95
3,Marie-France,France,COMPOUND_DECOMPOSITION,0.95
4,Sammie-Jo,Sammie,COMPOUND_DECOMPOSITION,0.95
5,Sammie-Jo,Jo,COMPOUND_DECOMPOSITION,0.95
6,Le-Anne,Le,COMPOUND_DECOMPOSITION,0.95
7,Le-Anne,Anne,COMPOUND_DECOMPOSITION,0.95
8,Marie-Ange,Marie,COMPOUND_DECOMPOSITION,0.95
9,Marie-Ange,Ange,COMPOUND_DECOMPOSITION,0.95


In [60]:
# ============================================================
# SECTION 11.2 - Verifying Compound Matrix
# ============================================================

print("Columns:")
print(compound_enhancement_matrix.columns.tolist())

print("\nUnique Original Names:")
print(compound_enhancement_matrix["ORIGINAL_NAME"].nunique())

Columns:
['ORIGINAL_NAME', 'CANONICAL_NAME', 'MATCH_TYPE', 'CONFIDENCE']

Unique Original Names:
11445


In [61]:
compound_enhancement_matrix.columns.tolist()

['ORIGINAL_NAME', 'CANONICAL_NAME', 'MATCH_TYPE', 'CONFIDENCE']

In [62]:
# ============================================================
# SECTION 11.3 - Summary
# ============================================================

compound_summary = pd.DataFrame({

    "Metric":[
        "Rows",
        "Unique Compound Names"
    ],

    "Value":[
        len(compound_enhancement_matrix),
        compound_enhancement_matrix["ORIGINAL_NAME"].nunique()
    ]

})

compound_summary

,Metric,Value
0,Rows,23275
1,Unique Compound Names,11445


# SECTION 12 – Candidate Generation

## Purpose

This section identifies the most likely canonical name candidates for unresolved names using approximate string matching.

Candidate generation is performed only after deterministic rule-based matching and compound name handling have been completed. This ensures that approximate matching is applied only where required, improving both efficiency and accuracy.

RapidFuzz is used to retrieve the highest-scoring candidate names for subsequent validation and confidence scoring.

In [63]:
# ============================================================
# SECTION 12.1 - RapidFuzz Candidate Generation
# ============================================================

from rapidfuzz import process, fuzz

In [64]:
# ============================================================
# SECTION 12.2 - Preparing Canonical Search List
# ============================================================

canonical_list = (
    canonical_clean["NORMALISED_NAME"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print(f"Unique canonical names: {len(canonical_list):,}")

Unique canonical names: 114,026


In [65]:
# ============================================================
# SECTION 12.3 - Building Search Index
# ============================================================

from collections import defaultdict

canonical_index = defaultdict(list)

for name in canonical_list:

    if pd.notna(name) and len(name) > 0:

        canonical_index[name[0]].append(name)

print(f"Search index created.")

print(f"Buckets: {len(canonical_index)}")

print("\nExample bucket sizes:")

for key in sorted(list(canonical_index.keys()))[:10]:

    print(f"{key}: {len(canonical_index[key]):,}")

Search index created.
Buckets: 26

Example bucket sizes:
A: 11,018
B: 4,060
C: 7,697
D: 6,445
E: 6,512
F: 2,264
G: 4,618
H: 3,410
I: 2,244
J: 11,218


In [66]:
# ============================================================
# SECTION 12.4 - Candidate Generator
# ============================================================

def get_best_candidate(name):

    if pd.isna(name):
        return None

    first_letter = name[0]

    candidates = canonical_index.get(first_letter, canonical_list)

    match = process.extractOne(

        query=name,

        choices=candidates,

        scorer=fuzz.WRatio,

        score_cutoff=85

    )

    return match

In [67]:
# ============================================================
# SECTION 12.5 - Generating Candidate Matches
# ============================================================

candidate_rows = []

remaining_names = (

    unknown_clean["STANDARDISED_NAME"]

    .dropna()

    .drop_duplicates()

)

total = len(remaining_names)

for i, name in enumerate(remaining_names, start=1):

    match = get_best_candidate(name)

    if match:

        candidate_rows.append({

            "Unknown_Name": name,

            "Candidate_Name": match[0],

            "Similarity_Score": match[1]

        })

    if i % 5000 == 0:

        print(f"Processed {i:,} of {total:,}")

candidate_df = pd.DataFrame(candidate_rows)

print("\nCandidate generation complete.")

print(f"Candidate matches: {len(candidate_df):,}")

Processed 5,000 of 66,875
Processed 10,000 of 66,875
Processed 15,000 of 66,875
Processed 20,000 of 66,875
Processed 25,000 of 66,875
Processed 30,000 of 66,875
Processed 35,000 of 66,875
Processed 40,000 of 66,875
Processed 45,000 of 66,875
Processed 50,000 of 66,875
Processed 55,000 of 66,875
Processed 60,000 of 66,875
Processed 65,000 of 66,875

Candidate generation complete.
Candidate matches: 34,174


In [68]:
candidate_df.shape

(34174, 3)

In [69]:
candidate_df.head(10)

,Unknown_Name,Candidate_Name,Similarity_Score
0,MATEUSZ,MAT,90.000000
1,BARTOSZ,BART,90.000000
2,PATRICIA,PATRICIA,100.000000
3,KINGA,KIN,90.000000
4,ALEXANDRU,ALEX,90.000000
5,TAHIRA,TAHIR,90.909091
6,PATRYK,PAT,90.000000
7,MIHAELA,MICHAELA,93.333333
8,MOHAMMED,MOHAMMED,100.000000
9,BARTLOMIEJ,BART,90.000000


In [70]:
candidate_df[
    candidate_df["Candidate_Name"].isin([
        "MAT",
        "PAT",
        "KIN",
        "BART",
        "ALEX"
    ])
]

,Unknown_Name,Candidate_Name,Similarity_Score
0,MATEUSZ,MAT,90.0
1,BARTOSZ,BART,90.0
3,KINGA,KIN,90.0
4,ALEXANDRU,ALEX,90.0
6,PATRYK,PAT,90.0
...,...,...,...
30979,PAT-TONY-STEWART,PAT,90.0
31076,ALEX-CHERYL,ALEX,90.0
31759,MAT-EMMA,MAT,90.0
32499,MAT-CHRISTINE,MAT,90.0


# SECTION 13 – Match Confidence

## Purpose

This section assigns a confidence level to each candidate match based on the similarity score produced by RapidFuzz.

The confidence score provides an interpretable measure of match reliability and supports downstream decision-making during SQL implementation.

Three confidence categories are used:

- High Confidence (≥95)
- Medium Confidence (90–94.99)
- Low Confidence (85–89.99)

In [71]:
# ============================================================
# SECTION 13.1 - Confidence Classification
# ============================================================

def confidence_level(score):

    if score >= 95:
        return "HIGH"

    elif score >= 90:
        return "MEDIUM"

    else:
        return "LOW"


candidate_df["Confidence"] = (
    candidate_df["Similarity_Score"]
    .apply(confidence_level)
)

print("Confidence scoring completed.")

Confidence scoring completed.


In [72]:
# ============================================================
# SECTION 13.2 - Confidence Summary
# ============================================================

confidence_summary = (
    candidate_df["Confidence"]
    .value_counts()
    .rename_axis("Confidence")
    .reset_index(name="Count")
)

confidence_summary

,Confidence,Count
0,MEDIUM,28918
1,LOW,4315
2,HIGH,941


In [73]:
candidate_df.head(10)

,Unknown_Name,Candidate_Name,Similarity_Score,Confidence
0,MATEUSZ,MAT,90.000000,MEDIUM
1,BARTOSZ,BART,90.000000,MEDIUM
2,PATRICIA,PATRICIA,100.000000,HIGH
3,KINGA,KIN,90.000000,MEDIUM
4,ALEXANDRU,ALEX,90.000000,MEDIUM
5,TAHIRA,TAHIR,90.909091,MEDIUM
6,PATRYK,PAT,90.000000,MEDIUM
7,MIHAELA,MICHAELA,93.333333,MEDIUM
8,MOHAMMED,MOHAMMED,100.000000,HIGH
9,BARTLOMIEJ,BART,90.000000,MEDIUM


# SECTION 14 – Final Match Matrix

## Purpose

This section constructs the final match matrix by combining candidate names, similarity scores and confidence classifications into a structured dataset.

The match matrix forms the final output of the name matching pipeline and is designed for straightforward integration into a relational SQL database.

In [74]:
# ============================================================
# SECTION 14.1 - Final Match Matrix
# ============================================================

final_match_matrix = candidate_df.copy()

final_match_matrix = final_match_matrix.rename(columns={

    "Unknown_Name":"ORIGINAL_NAME",

    "Candidate_Name":"CANONICAL_NAME",

    "Similarity_Score":"CONFIDENCE_SCORE"

})

print("Final Match Matrix created successfully.")

print(f"Rows : {len(final_match_matrix):,}")

print(f"Columns : {len(final_match_matrix.columns)}")

Final Match Matrix created successfully.
Rows : 34,174
Columns : 4


In [75]:
# ============================================================
# SECTION 14.2 - Match Type
# ============================================================

final_match_matrix["MATCH_TYPE"] = "FUZZY_MATCH"

cols = [

    "ORIGINAL_NAME",

    "CANONICAL_NAME",

    "MATCH_TYPE",

    "CONFIDENCE_SCORE",

    "Confidence"

]

final_match_matrix = final_match_matrix[cols]

final_match_matrix.head(20)

,ORIGINAL_NAME,CANONICAL_NAME,MATCH_TYPE,CONFIDENCE_SCORE,Confidence
0,MATEUSZ,MAT,FUZZY_MATCH,90.000000,MEDIUM
1,BARTOSZ,BART,FUZZY_MATCH,90.000000,MEDIUM
2,PATRICIA,PATRICIA,FUZZY_MATCH,100.000000,HIGH
3,KINGA,KIN,FUZZY_MATCH,90.000000,MEDIUM
4,ALEXANDRU,ALEX,FUZZY_MATCH,90.000000,MEDIUM
5,TAHIRA,TAHIR,FUZZY_MATCH,90.909091,MEDIUM
6,PATRYK,PAT,FUZZY_MATCH,90.000000,MEDIUM
7,MIHAELA,MICHAELA,FUZZY_MATCH,93.333333,MEDIUM
8,MOHAMMED,MOHAMMED,FUZZY_MATCH,100.000000,HIGH
9,BARTLOMIEJ,BART,FUZZY_MATCH,90.000000,MEDIUM


In [76]:
# ============================================================
# SECTION 14.3 - Match Matrix Summary
# ============================================================

summary = pd.DataFrame({

    "Metric":[

        "Total Matches",

        "Unique Original Names",

        "Unique Canonical Names"

    ],

    "Value":[

        len(final_match_matrix),

        final_match_matrix["ORIGINAL_NAME"].nunique(),

        final_match_matrix["CANONICAL_NAME"].nunique()

    ]

})

summary

,Metric,Value
0,Total Matches,34174
1,Unique Original Names,34174
2,Unique Canonical Names,7455


In [77]:
# ============================================================
# SECTION 14.4 - Match Quality Assessment
# ============================================================

def classify_match(row):

    original = str(row["ORIGINAL_NAME"]).strip().upper()
    canonical = str(row["CANONICAL_NAME"]).strip().upper()
    score = row["CONFIDENCE_SCORE"]

    # Exact match
    if original == canonical:
        return "AUTO_APPROVED"

    # Very high confidence
    if score >= 95:
        return "AUTO_APPROVED"

    # Good fuzzy match
    if score >= 90:
        return "REVIEW"

    # Everything else
    return "MANUAL_REVIEW"


final_match_matrix["MATCH_STATUS"] = (
    final_match_matrix.apply(classify_match, axis=1)
)

print("Match status assigned successfully.")

Match status assigned successfully.


In [78]:
# ============================================================
# SECTION 14.5 - Match Status Summary
# ============================================================

status_summary = (
    final_match_matrix["MATCH_STATUS"]
    .value_counts()
    .rename_axis("Match Status")
    .reset_index(name="Count")
)

status_summary

,Match Status,Count
0,REVIEW,28918
1,MANUAL_REVIEW,4315
2,AUTO_APPROVED,941


In [79]:
# ============================================================
# SECTION 14.6 - SQL Ready Match Matrix
# ============================================================

sql_ready_match_matrix = final_match_matrix.copy()

sql_ready_match_matrix = sql_ready_match_matrix[[
    "ORIGINAL_NAME",
    "CANONICAL_NAME",
    "MATCH_TYPE",
    "CONFIDENCE_SCORE",
    "Confidence",
    "MATCH_STATUS"
]]

sql_ready_match_matrix.head(20)

,ORIGINAL_NAME,CANONICAL_NAME,MATCH_TYPE,CONFIDENCE_SCORE,Confidence,MATCH_STATUS
0,MATEUSZ,MAT,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
1,BARTOSZ,BART,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
2,PATRICIA,PATRICIA,FUZZY_MATCH,100.000000,HIGH,AUTO_APPROVED
3,KINGA,KIN,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
4,ALEXANDRU,ALEX,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
5,TAHIRA,TAHIR,FUZZY_MATCH,90.909091,MEDIUM,REVIEW
6,PATRYK,PAT,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
7,MIHAELA,MICHAELA,FUZZY_MATCH,93.333333,MEDIUM,REVIEW
8,MOHAMMED,MOHAMMED,FUZZY_MATCH,100.000000,HIGH,AUTO_APPROVED
9,BARTLOMIEJ,BART,FUZZY_MATCH,90.000000,MEDIUM,REVIEW


In [80]:
# ============================================================
# SECTION 14.7 - Accepted SQL Match Matrix
# ============================================================

accepted_sql_matches = sql_ready_match_matrix[
    sql_ready_match_matrix["MATCH_STATUS"] != "MANUAL_REVIEW"
].reset_index(drop=True)

print(f"Accepted SQL matches : {len(accepted_sql_matches):,}")

accepted_sql_matches.head(20)

Accepted SQL matches : 29,859


,ORIGINAL_NAME,CANONICAL_NAME,MATCH_TYPE,CONFIDENCE_SCORE,Confidence,MATCH_STATUS
0,MATEUSZ,MAT,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
1,BARTOSZ,BART,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
2,PATRICIA,PATRICIA,FUZZY_MATCH,100.000000,HIGH,AUTO_APPROVED
3,KINGA,KIN,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
4,ALEXANDRU,ALEX,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
5,TAHIRA,TAHIR,FUZZY_MATCH,90.909091,MEDIUM,REVIEW
6,PATRYK,PAT,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
7,MIHAELA,MICHAELA,FUZZY_MATCH,93.333333,MEDIUM,REVIEW
8,MOHAMMED,MOHAMMED,FUZZY_MATCH,100.000000,HIGH,AUTO_APPROVED
9,BARTLOMIEJ,BART,FUZZY_MATCH,90.000000,MEDIUM,REVIEW


In [81]:
# ============================================================
# SECTION 14.8 - Manual Review Queue
# ============================================================

manual_review_queue = sql_ready_match_matrix[
    sql_ready_match_matrix["MATCH_STATUS"] == "MANUAL_REVIEW"
].reset_index(drop=True)

print(f"Manual review records : {len(manual_review_queue):,}")

manual_review_queue.head(20)

Manual review records : 4,315


,ORIGINAL_NAME,CANONICAL_NAME,MATCH_TYPE,CONFIDENCE_SCORE,Confidence,MATCH_STATUS
0,FARHANA,FARZANA,FUZZY_MATCH,85.714286,LOW,MANUAL_REVIEW
1,ABDI,ABI,FUZZY_MATCH,85.714286,LOW,MANUAL_REVIEW
2,DIMITRIOS,DIMITRI,FUZZY_MATCH,87.500000,LOW,MANUAL_REVIEW
3,SHABNAM,SHABANA,FUZZY_MATCH,85.714286,LOW,MANUAL_REVIEW
4,GHEORGHE,GEORGE,FUZZY_MATCH,85.714286,LOW,MANUAL_REVIEW
5,ABUL,ABDUL,FUZZY_MATCH,88.888889,LOW,MANUAL_REVIEW
6,IRAM,IRA,FUZZY_MATCH,85.714286,LOW,MANUAL_REVIEW
7,PRAVEEN,PARVEEN,FUZZY_MATCH,85.714286,LOW,MANUAL_REVIEW
8,IOANA,IONA,FUZZY_MATCH,88.888889,LOW,MANUAL_REVIEW
9,BALWINDER,BALJINDER,FUZZY_MATCH,88.888889,LOW,MANUAL_REVIEW


# SECTION 15 – SQL Export

## Purpose

This section exports the final outputs of the name matching pipeline into SQL-compatible CSV files.

Three outputs are generated:

- Accepted SQL Match Matrix
- Manual Review Queue
- Complete Match Matrix

These outputs can be imported directly into relational database tables or used during the SQL implementation phase.

In [105]:
print("Available DataFrames:\n")

for name in list(globals()):
    obj = globals()[name]

    if hasattr(obj, "shape"):
        try:
            print(f"{name:35} {obj.shape}")
        except:
            pass

Available DataFrames:

np                                  <function shape at 0x000002026586E480>
forenames                           (204108, 9)
surnames                            (4153336, 7)
_5                                  (5, 9)
_6                                  (5, 7)
df_forenames                        (204108, 13)
df_surnames                         (4153336, 11)
_9                                  (5, 11)
_10                                 (2, 4)
summary                             (3, 2)
missing_forenames                   (9, 1)
missing_surnames                    (7, 1)
duplicates                          (2, 2)
_13                                 (2, 2)
memory_usage                        (2, 4)
forename_lengths                    (204108,)
surname_lengths                     (4153336,)
forename_flags                      (4, 3)
canonical_names                     (114027, 9)
accepted_variants                   (22933, 9)
unknown_names                       (66878, 

In [107]:
import os

# Export NAME_MASTER

# Create the directory if it doesn't exist
output_dir = r"C:\Temp"
os.makedirs(output_dir, exist_ok=True)  # Creates directory if it doesn't exist

name_master.to_csv(
    r"C:\Temp\NAME_MASTER.csv",
    index=False,
    encoding="utf-8-sig"
)

print("NAME_MASTER exported successfully.")

NAME_MASTER exported successfully.


In [109]:
name_master.head()

,CANONICAL_NAME,NAME_TYPE
0,John,FORENAME
1,David,FORENAME
2,James,FORENAME
3,Michael,FORENAME
4,Paul,FORENAME


In [111]:
# ==========================================================
# SECTION 15.1C - Export NAME_VARIANT
# ==========================================================

name_variant.to_csv(
    r"C:\Temp\NAME_VARIANT.csv",
    index=False,
    encoding="utf-8-sig"
)

print("NAME_VARIANT exported successfully.")
print(name_variant.shape)
print(name_variant.columns.tolist())

NAME_VARIANT exported successfully.
(114027, 5)
['ORIGINAL_NAME', 'CANONICAL_NAME', 'MATCH_TYPE', 'EDIT_DISTANCE', 'CONFIDENCE_SCORE']


In [110]:
name_variant.head()

,ORIGINAL_NAME,CANONICAL_NAME,MATCH_TYPE,EDIT_DISTANCE,CONFIDENCE_SCORE
0,JOHN,John,NaN,0,1.0
1,DAVID,David,NaN,0,1.0
2,JAMES,James,NaN,0,1.0
3,MICHAEL,Michael,NaN,0,1.0
4,PAUL,Paul,NaN,0,1.0


In [82]:
# ============================================================
# SECTION 15.1 - Exporting SQL Files
# ============================================================

accepted_sql_matches.to_csv(
    "accepted_sql_matches.csv",
    index=False
)

manual_review_queue.to_csv(
    "manual_review_queue.csv",
    index=False
)

sql_ready_match_matrix.to_csv(
    "complete_match_matrix.csv",
    index=False
)

print("SQL export completed successfully.\n")

print(f"Accepted matches : {len(accepted_sql_matches):,}")
print(f"Manual review    : {len(manual_review_queue):,}")
print(f"Complete matrix  : {len(sql_ready_match_matrix):,}")

SQL export completed successfully.

Accepted matches : 29,859
Manual review    : 4,315
Complete matrix  : 34,174


In [102]:
print(canonical_names.shape)
print(accepted_variants.shape)

print(canonical_names.columns.tolist())
print(accepted_variants.columns.tolist())

(114027, 9)
(22933, 9)
['FORENAME_UPPER', 'FORENAME_PROPER', 'FORENAME_MATCH', 'EDIT_DISTANCE', 'JARO_WINKLER', 'FORENAME_FLAG', 'AMOUNT', 'GENDER', 'GENDER_COUNT']
['FORENAME_UPPER', 'FORENAME_PROPER', 'FORENAME_MATCH', 'EDIT_DISTANCE', 'JARO_WINKLER', 'FORENAME_FLAG', 'AMOUNT', 'GENDER', 'GENDER_COUNT']


In [103]:
# ==========================================================
# SECTION 15.1A - Export NAME_MASTER
# ==========================================================

import os

sql_export_path = r"C:\Users\omoto\OneDrive\Desktop\MSc DISSERTATION\PDSA Name Matching\SQL_EXPORTS"

os.makedirs(sql_export_path, exist_ok=True)

name_master = canonical_names[["FORENAME_PROPER"]].copy()

name_master.columns = ["CANONICAL_NAME"]

name_master["NAME_TYPE"] = "FORENAME"

name_master.to_csv(
    os.path.join(sql_export_path, "NAME_MASTER.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("NAME_MASTER exported successfully.")
print(name_master.shape)

display(name_master.head())

NAME_MASTER exported successfully.
(114027, 2)


,CANONICAL_NAME,NAME_TYPE
0,John,FORENAME
1,David,FORENAME
2,James,FORENAME
3,Michael,FORENAME
4,Paul,FORENAME


In [104]:
# ==========================================================
# SECTION 15.1B - Export NAME_VARIANT
# ==========================================================

name_variant = canonical_names[
    ["FORENAME_UPPER",
     "FORENAME_PROPER",
     "FORENAME_MATCH",
     "EDIT_DISTANCE",
     "JARO_WINKLER"]
].copy()

name_variant.columns = [
    "ORIGINAL_NAME",
    "CANONICAL_NAME",
    "MATCH_TYPE",
    "EDIT_DISTANCE",
    "CONFIDENCE_SCORE"
]

name_variant.to_csv(
    os.path.join(sql_export_path, "NAME_VARIANT.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("NAME_VARIANT exported successfully.")
print(name_variant.shape)

display(name_variant.head())

NAME_VARIANT exported successfully.
(114027, 5)


,ORIGINAL_NAME,CANONICAL_NAME,MATCH_TYPE,EDIT_DISTANCE,CONFIDENCE_SCORE
0,JOHN,John,NaN,0,1.0
1,DAVID,David,NaN,0,1.0
2,JAMES,James,NaN,0,1.0
3,MICHAEL,Michael,NaN,0,1.0
4,PAUL,Paul,NaN,0,1.0


In [83]:
# ============================================================
# SECTION 15.2 - Export Summary
# ============================================================

export_summary = pd.DataFrame({

    "Dataset":[
        "Accepted SQL Matches",
        "Manual Review Queue",
        "Complete Match Matrix"
    ],

    "Records":[
        len(accepted_sql_matches),
        len(manual_review_queue),
        len(sql_ready_match_matrix)
    ]

})

export_summary

,Dataset,Records
0,Accepted SQL Matches,29859
1,Manual Review Queue,4315
2,Complete Match Matrix,34174


In [84]:
# ============================================================
# SECTION 15.3 - Verifing SQL Structure
# ============================================================

print("Accepted SQL Matches")
display(accepted_sql_matches.info())

print("\nManual Review Queue")
display(manual_review_queue.info())

print("\nComplete Match Matrix")
display(sql_ready_match_matrix.info())

Accepted SQL Matches
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29859 entries, 0 to 29858
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORIGINAL_NAME     29859 non-null  object 
 1   CANONICAL_NAME    29859 non-null  object 
 2   MATCH_TYPE        29859 non-null  object 
 3   CONFIDENCE_SCORE  29859 non-null  float64
 4   Confidence        29859 non-null  object 
 5   MATCH_STATUS      29859 non-null  object 
dtypes: float64(1), object(5)
memory usage: 1.4+ MB


None


Manual Review Queue
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4315 entries, 0 to 4314
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORIGINAL_NAME     4315 non-null   object 
 1   CANONICAL_NAME    4315 non-null   object 
 2   MATCH_TYPE        4315 non-null   object 
 3   CONFIDENCE_SCORE  4315 non-null   float64
 4   Confidence        4315 non-null   object 
 5   MATCH_STATUS      4315 non-null   object 
dtypes: float64(1), object(5)
memory usage: 202.4+ KB


None


Complete Match Matrix
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34174 entries, 0 to 34173
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORIGINAL_NAME     34174 non-null  object 
 1   CANONICAL_NAME    34174 non-null  object 
 2   MATCH_TYPE        34174 non-null  object 
 3   CONFIDENCE_SCORE  34174 non-null  float64
 4   Confidence        34174 non-null  object 
 5   MATCH_STATUS      34174 non-null  object 
dtypes: float64(1), object(5)
memory usage: 1.6+ MB


None

# SECTION 16 – Performance Evaluation

## Purpose

This section evaluates the overall performance of the proposed name matching framework.

Key performance indicators include:

- Rule-based matching performance
- RapidFuzz matching performance
- Match confidence distribution
- Manual review rate
- Overall processing statistics

These metrics provide quantitative evidence of the effectiveness of the proposed framework.

In [85]:
# ============================================================
# SECTION 16.1 - Overall Pipeline Statistics
# ============================================================

pipeline_summary = pd.DataFrame({

    "Stage":[

        "Canonical Names",

        "Accepted Variants",

        "Unknown Names",

        "Rule-based Matches",

        "RapidFuzz Matches",

        "Auto Approved",

        "Review",

        "Manual Review"

    ],

    "Count":[

        len(canonical_names),

        len(accepted_variants),

        len(unknown_names),

        unknown_clean["RULE_MATCH"].notna().sum(),

        len(candidate_df),

        (sql_ready_match_matrix["MATCH_STATUS"]=="AUTO_APPROVED").sum(),

        (sql_ready_match_matrix["MATCH_STATUS"]=="REVIEW").sum(),

        (sql_ready_match_matrix["MATCH_STATUS"]=="MANUAL_REVIEW").sum()

    ]

})

pipeline_summary

,Stage,Count
0,Canonical Names,114027
1,Accepted Variants,22933
2,Unknown Names,66878
3,Rule-based Matches,3
4,RapidFuzz Matches,34174
5,Auto Approved,941
6,Review,28918
7,Manual Review,4315


In [86]:
# ============================================================
# SECTION 16.2 - Match Quality Distribution
# ============================================================

quality_distribution = (

    sql_ready_match_matrix["MATCH_STATUS"]

    .value_counts(normalize=True)

    .mul(100)

    .round(2)

    .rename_axis("Match Status")

    .reset_index(name="Percentage")

)

quality_distribution

,Match Status,Percentage
0,REVIEW,84.62
1,MANUAL_REVIEW,12.63
2,AUTO_APPROVED,2.75


In [87]:
# ============================================================
# SECTION 16.3 - Confidence Distribution
# ============================================================

confidence_distribution = (

    sql_ready_match_matrix["Confidence"]

    .value_counts(normalize=True)

    .mul(100)

    .round(2)

    .rename_axis("Confidence")

    .reset_index(name="Percentage")

)

confidence_distribution

,Confidence,Percentage
0,MEDIUM,84.62
1,LOW,12.63
2,HIGH,2.75


In [88]:
# ============================================================
# SECTION 16.4 - Overall Performance Metrics
# ============================================================

performance_metrics = pd.DataFrame({

    "Metric":[

        "Total Candidate Matches",

        "Accepted SQL Matches",

        "Manual Review Queue",

        "Unique Canonical Names",

        "Unique Original Names"

    ],

    "Value":[

        len(candidate_df),

        len(accepted_sql_matches),

        len(manual_review_queue),

        sql_ready_match_matrix["CANONICAL_NAME"].nunique(),

        sql_ready_match_matrix["ORIGINAL_NAME"].nunique()

    ]

})

performance_metrics

,Metric,Value
0,Total Candidate Matches,34174
1,Accepted SQL Matches,29859
2,Manual Review Queue,4315
3,Unique Canonical Names,7455
4,Unique Original Names,34174


## Evaluation Summary

The proposed framework successfully combines deterministic rule-based matching with fuzzy matching techniques to identify candidate canonical names.

The introduction of confidence scoring and match-status classification enables automatic acceptance of high-confidence matches while routing uncertain matches for manual review.

This hybrid strategy improves transparency and provides a practical workflow for integration into SQL-based master data management systems.

The resulting outputs demonstrate the feasibility of combining rule-based and similarity-based techniques within a scalable name standardisation framework.

# SECTION 17 – Project Summary and Conclusions

## Purpose

This section summarises the overall performance of the proposed name matching framework and evaluates its suitability for deployment within a SQL-based environment.

The framework demonstrates a hybrid approach that combines deterministic rule-based matching with similarity-based candidate generation using RapidFuzz. The integration of confidence scoring and match-status classification provides an auditable decision process suitable for master data management applications.

The final outputs consist of:

- SQL-ready accepted matches
- Manual review queue
- Complete match matrix

These outputs provide a structured foundation for implementation within relational database systems while supporting future enhancements through additional business rules and expanded variant dictionaries.

In [89]:
# ============================================================
# SECTION 17.1 - Final Project Statistics
# ============================================================

final_statistics = pd.DataFrame({

    "Project Metric":[

        "Forename Records",

        "Surname Records",

        "Canonical Names",

        "Accepted Variants",

        "Unknown Names",

        "RapidFuzz Candidate Matches",

        "Accepted SQL Matches",

        "Manual Review Queue",

        "Complete Match Matrix"

    ],

    "Value":[

        len(forenames),

        len(surnames),

        len(canonical_names),

        len(accepted_variants),

        len(unknown_names),

        len(candidate_df),

        len(accepted_sql_matches),

        len(manual_review_queue),

        len(sql_ready_match_matrix)

    ]

})

final_statistics

,Project Metric,Value
0,Forename Records,204108
1,Surname Records,4153336
2,Canonical Names,114027
3,Accepted Variants,22933
4,Unknown Names,66878
5,RapidFuzz Candidate Matches,34174
6,Accepted SQL Matches,29859
7,Manual Review Queue,4315
8,Complete Match Matrix,34174


In [90]:
# ============================================================
# SECTION 17.2 - Pipeline Stages
# ============================================================

pipeline = pd.DataFrame({

    "Stage":[

        "1. Data Loading",

        "2. Data Profiling",

        "3. Data Cleaning",

        "4. Name Normalisation",

        "5. Rule-Based Matching",

        "6. Compound Name Enhancement",

        "7. RapidFuzz Candidate Generation",

        "8. Confidence Classification",

        "9. Match Quality Assessment",

        "10. SQL Export"

    ],

    "Status":[

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed"

    ]

})

pipeline

,Stage,Status
0,1. Data Loading,Completed
1,2. Data Profiling,Completed
2,3. Data Cleaning,Completed
3,4. Name Normalisation,Completed
4,5. Rule-Based Matching,Completed
5,6. Compound Name Enhancement,Completed
6,7. RapidFuzz Candidate Generation,Completed
7,8. Confidence Classification,Completed
8,9. Match Quality Assessment,Completed
9,10. SQL Export,Completed


In [91]:
# ============================================================
# SECTION 17.3 - Project Deliverables
# ============================================================

deliverables = pd.DataFrame({

    "Deliverable":[

        "Cleaned Forename Dataset",

        "Canonical Name Dataset",

        "Accepted Variant Dictionary",

        "Compound Enhancement Matrix",

        "RapidFuzz Candidate Matches",

        "Final Match Matrix",

        "Accepted SQL Match Matrix",

        "Manual Review Queue",

        "SQL Export Files"

    ],

    "Status":[

        "Complete",

        "Complete",

        "Complete",

        "Complete",

        "Complete",

        "Complete",

        "Complete",

        "Complete",

        "Complete"

    ]

})

deliverables

,Deliverable,Status
0,Cleaned Forename Dataset,Complete
1,Canonical Name Dataset,Complete
2,Accepted Variant Dictionary,Complete
3,Compound Enhancement Matrix,Complete
4,RapidFuzz Candidate Matches,Complete
5,Final Match Matrix,Complete
6,Accepted SQL Match Matrix,Complete
7,Manual Review Queue,Complete
8,SQL Export Files,Complete


# Final Conclusion

The developed framework successfully demonstrates an end-to-end name matching and standardisation pipeline suitable for SQL implementation.

The proposed solution integrates data cleaning, name normalisation, deterministic rule-based matching, compound name handling and similarity-based candidate generation to improve the consistency of personal name records.

The inclusion of confidence scoring together with manual review classification provides an auditable decision process that reduces the risk of incorrect automatic mappings while supporting scalable master data management.

Although some low-confidence matches remain dependent on manual verification, the framework establishes a practical foundation that can be extended through richer variant dictionaries, additional linguistic rules and organisation-specific business logic.

In [92]:
accepted_sql_matches.head(20)

,ORIGINAL_NAME,CANONICAL_NAME,MATCH_TYPE,CONFIDENCE_SCORE,Confidence,MATCH_STATUS
0,MATEUSZ,MAT,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
1,BARTOSZ,BART,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
2,PATRICIA,PATRICIA,FUZZY_MATCH,100.000000,HIGH,AUTO_APPROVED
3,KINGA,KIN,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
4,ALEXANDRU,ALEX,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
5,TAHIRA,TAHIR,FUZZY_MATCH,90.909091,MEDIUM,REVIEW
6,PATRYK,PAT,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
7,MIHAELA,MICHAELA,FUZZY_MATCH,93.333333,MEDIUM,REVIEW
8,MOHAMMED,MOHAMMED,FUZZY_MATCH,100.000000,HIGH,AUTO_APPROVED
9,BARTLOMIEJ,BART,FUZZY_MATCH,90.000000,MEDIUM,REVIEW


In [93]:
manual_review_queue.head(20)

,ORIGINAL_NAME,CANONICAL_NAME,MATCH_TYPE,CONFIDENCE_SCORE,Confidence,MATCH_STATUS
0,FARHANA,FARZANA,FUZZY_MATCH,85.714286,LOW,MANUAL_REVIEW
1,ABDI,ABI,FUZZY_MATCH,85.714286,LOW,MANUAL_REVIEW
2,DIMITRIOS,DIMITRI,FUZZY_MATCH,87.500000,LOW,MANUAL_REVIEW
3,SHABNAM,SHABANA,FUZZY_MATCH,85.714286,LOW,MANUAL_REVIEW
4,GHEORGHE,GEORGE,FUZZY_MATCH,85.714286,LOW,MANUAL_REVIEW
5,ABUL,ABDUL,FUZZY_MATCH,88.888889,LOW,MANUAL_REVIEW
6,IRAM,IRA,FUZZY_MATCH,85.714286,LOW,MANUAL_REVIEW
7,PRAVEEN,PARVEEN,FUZZY_MATCH,85.714286,LOW,MANUAL_REVIEW
8,IOANA,IONA,FUZZY_MATCH,88.888889,LOW,MANUAL_REVIEW
9,BALWINDER,BALJINDER,FUZZY_MATCH,88.888889,LOW,MANUAL_REVIEW


In [94]:
sql_ready_match_matrix.head(20)

,ORIGINAL_NAME,CANONICAL_NAME,MATCH_TYPE,CONFIDENCE_SCORE,Confidence,MATCH_STATUS
0,MATEUSZ,MAT,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
1,BARTOSZ,BART,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
2,PATRICIA,PATRICIA,FUZZY_MATCH,100.000000,HIGH,AUTO_APPROVED
3,KINGA,KIN,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
4,ALEXANDRU,ALEX,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
5,TAHIRA,TAHIR,FUZZY_MATCH,90.909091,MEDIUM,REVIEW
6,PATRYK,PAT,FUZZY_MATCH,90.000000,MEDIUM,REVIEW
7,MIHAELA,MICHAELA,FUZZY_MATCH,93.333333,MEDIUM,REVIEW
8,MOHAMMED,MOHAMMED,FUZZY_MATCH,100.000000,HIGH,AUTO_APPROVED
9,BARTLOMIEJ,BART,FUZZY_MATCH,90.000000,MEDIUM,REVIEW


In [95]:
accepted_sql_matches.to_csv(
    r"C:\Users\omoto\OneDrive\Desktop\MSc DISSERTATION\PDSA Name Matching\accepted_sql_matches.csv",
    index=False
)
manual_review_queue.to_csv(
    r"C:\Users\omoto\OneDrive\Desktop\MSc DISSERTATION\PDSA Name Matching\manual_review_queue.csv",
    index=False
)


sql_ready_match_matrix.to_csv(
    r"C:\Users\omoto\OneDrive\Desktop\MSc DISSERTATION\PDSA Name Matching\complete_match_matrix.csv",
    index=False
)

print("CSV export completed.")

CSV export completed.


In [96]:
print("Existing variables:")

for name in sorted(globals()):
    if not name.startswith("_"):
        obj = globals()[name]
        if hasattr(obj, "shape"):
            print(f"{name:<30} {obj.shape}")

Existing variables:
accepted_clean                 (22933, 11)
accepted_sql_matches           (29859, 6)
accepted_variants              (22933, 9)
apostrophe_summary             (3, 2)
ascii_summary                  (3, 2)
candidate_df                   (34174, 4)
canonical_clean                (114027, 11)
canonical_names                (114027, 9)
clean_forenames                (204081, 14)
clean_surnames                 (4153176, 12)
cleaning_summary               (3, 3)
compound_enhancement_matrix    (23275, 4)
compound_mask                  (66878,)
compound_summary               (2, 2)
confidence_distribution        (3, 2)
confidence_summary             (3, 2)
deliverables                   (9, 2)
df_forenames                   (204108, 13)
df_surnames                    (4153336, 11)
duplicates                     (2, 2)
export_summary                 (3, 2)
final_match_matrix             (34174, 6)
final_statistics               (9, 2)
forename_flags                 (4, 3)
fore

In [97]:
# ==========================================================
# FINAL PROJECT DASHBOARD
# ==========================================================

project_dashboard = pd.DataFrame({

    "Metric":[

        "Original Forenames",

        "Original Surnames",

        "Valid Forenames",

        "Valid Surnames",

        "Canonical Dictionary",

        "Accepted Variants",

        "Unknown Names",

        "Rejected Names"

    ],

    "Value":[

        len(forenames),

        len(surnames),

        len(clean_forenames),

        len(clean_surnames),

        len(canonical_names),

        len(accepted_variants),

        len(unknown_names),

        len(rejected_names)

    ]

})

display(project_dashboard)

,Metric,Value
0,Original Forenames,204108
1,Original Surnames,4153336
2,Valid Forenames,204081
3,Valid Surnames,4153176
4,Canonical Dictionary,114027
5,Accepted Variants,22933
6,Unknown Names,66878
7,Rejected Names,270


In [98]:
#==Export

project_dashboard.to_csv(
    "Project_Dashboard.csv",
    index=False
)

print("✓ Project Dashboard Exported")

✓ Project Dashboard Exported


In [99]:
#== Pipeline Validation

print("="*60)

print("PIPELINE VALIDATION")

print("="*60)

assert len(clean_forenames)<=len(forenames)

assert len(clean_surnames)<=len(surnames)

assert len(canonical_names)>=len(accepted_variants)

assert len(unknown_names)>=len(rejected_names)

print("✓ Validation Passed")

PIPELINE VALIDATION
✓ Validation Passed


In [100]:
#==Final SQL Export summary

export_summary = pd.DataFrame({

"Export File":[

"Canonical_Names.csv",

"Accepted_Variants.csv",

"Unknown_Names.csv",

"Rejected_Names.csv",

"Data_Quality_Dashboard.csv",

"Project_Dashboard.csv"

],

"Rows":[

len(canonical_names),

len(accepted_variants),

len(unknown_names),

len(rejected_names),

len(quality_dashboard),

len(project_dashboard)

]

})

display(export_summary)

,Export File,Rows
0,Canonical_Names.csv,114027
1,Accepted_Variants.csv,22933
2,Unknown_Names.csv,66878
3,Rejected_Names.csv,270
4,Data_Quality_Dashboard.csv,2
5,Project_Dashboard.csv,8


In [101]:
#==Export

export_summary.to_csv(
    "Export_Summary.csv",
    index=False
)

print("✓ Export Summary Created")

✓ Export Summary Created
